# dYdX v4 Markets Discovery

Explore all available perpetual markets on dYdX v4 chain.

## About dYdX
- **Type**: Cosmos-based blockchain (dYdX v4 launched 2023)
- **Decentralized**: Fully on-chain order book and settlement
- **Available**: Globally (no KYC, no geo-restrictions)
- **Settlement**: USDC (cross-margined)
- **Funding**: Hourly (charged at XX:00:00 UTC)

## Funding Rate Formula
- **Funding Rate = (Premium Component / 8) + Interest Rate**
- Premium: TWAP of mark-index spread over 1 hour
- Interest Rate: 0% for cross markets, 0.125 bps/hour for isolated
- Charged **every hour** (not every 8 hours!)

## API Details
- **Mainnet**: https://indexer.dydx.trade/v4
- **Market Format**: BTC-USD, ETH-USD (hyphenated)
- **Timezone**: UTC

In [1]:
import requests
import pandas as pd
from datetime import datetime

BASE_URL = "https://indexer.dydx.trade/v4"

## Check Server Time and Timezone

In [2]:
# Get server time
response = requests.get(f"{BASE_URL}/time")
server_time = response.json()
print(f"Server time: {server_time}")

# Get current height
response = requests.get(f"{BASE_URL}/height")
height_data = response.json()
print(f"\nCurrent block: {height_data}")

Server time: {'iso': '2026-03-01T18:55:10.188Z', 'epoch': 1772391310.188}

Current block: {'height': '78154692', 'time': '2026-03-01T18:55:09.693Z'}


## Discover All Perpetual Markets

In [3]:
# Get all perpetual markets
response = requests.get(f"{BASE_URL}/perpetualMarkets")
markets_data = response.json()

print(f"Total markets: {len(markets_data['markets'])}")
print(f"\nFirst market example:")
first_market = list(markets_data['markets'].values())[0]
for key, value in first_market.items():
    print(f"  {key}: {value}")

Total markets: 293

First market example:
  clobPairId: 0
  ticker: BTC-USD
  status: ACTIVE
  oraclePrice: 66284.69662
  priceChange24H: 319.83713
  volume24H: 77185585.2006
  trades24H: 35438
  nextFundingRate: -0.00002081818181818182
  initialMarginFraction: 0.02
  maintenanceMarginFraction: 0.012
  openInterest: 295.5872
  atomicResolution: -10
  quantumConversionExponent: -9
  tickSize: 1
  stepSize: 0.0001
  stepBaseQuantums: 1000000
  subticksPerTick: 100000
  marketType: CROSS
  openInterestLowerCap: 0
  openInterestUpperCap: 0
  baseOpenInterest: 782.0931
  defaultFundingRate1H: 0


## Parse Market Data

In [4]:
markets = []
for ticker, market in markets_data['markets'].items():
    markets.append({
        'ticker': ticker,
        'base_asset': market.get('baseAsset', ''),
        'quote_asset': market.get('quoteAsset', ''),
        'price': float(market.get('oraclePrice', 0)),
        'volume_24h': float(market.get('volume24H', 0)),
        'open_interest': float(market.get('openInterestUSDC', 0)) / 1e6,  # Convert to USDC
        'trades_24h': int(market.get('trades24H', 0)),
        'next_funding_rate': float(market.get('nextFundingRate', 0)),
        'status': market.get('status', ''),
    })

df_markets = pd.DataFrame(markets)
df_markets = df_markets.sort_values('volume_24h', ascending=False)

print(f"Total markets: {len(df_markets)}")
print(f"\nMarket status distribution:")
print(df_markets['status'].value_counts())

Total markets: 293

Market status distribution:
status
ACTIVE              205
FINAL_SETTLEMENT     88
Name: count, dtype: int64


## Top Markets by Volume

In [ ]:
# Show top 50 markets by volume
top_markets = df_markets.head(50)

print("Top 50 Markets by 24h Volume:")
print("=" * 100)
for idx, row in top_markets.iterrows():
    print(f"{row['ticker']:<15} | Vol: ${row['volume_24h']/1e6:>8.1f}M | OI: ${row['open_interest']:>8.1f}M | "
    f"Price: ${row['price']:>10.2f} | Funding: {row['next_funding_rate']*100:>6.4f}%")

Top 50 Markets by 24h Volume:
BTC-USD         | Vol: $    77.2M | OI: $     0.0M | Price: $  66284.70 | Funding: -0.0021%
ETH-USD         | Vol: $    24.4M | OI: $     0.0M | Price: $   1975.75 | Funding: -0.0026%
SOL-USD         | Vol: $    11.0M | OI: $     0.0M | Price: $     83.92 | Funding: -0.0028%
ADA-USD         | Vol: $     0.5M | OI: $     0.0M | Price: $      0.28 | Funding: 0.0000%
NEAR-USD        | Vol: $     0.4M | OI: $     0.0M | Price: $      1.15 | Funding: 0.0000%
PAXG-USD        | Vol: $     0.3M | OI: $     0.0M | Price: $   5403.24 | Funding: 0.0109%
XRP-USD         | Vol: $     0.3M | OI: $     0.0M | Price: $      1.37 | Funding: -0.0001%
HYPE-USD        | Vol: $     0.3M | OI: $     0.0M | Price: $     31.75 | Funding: 0.0000%
AVAX-USD        | Vol: $     0.3M | OI: $     0.0M | Price: $      9.04 | Funding: 0.0000%
DOT-USD         | Vol: $     0.3M | OI: $     0.0M | Price: $      1.53 | Funding: 0.0000%
LINK-USD        | Vol: $     0.2M | OI: $     0.0M | Pri

In [8]:
top_markets

,ticker,base_asset,quote_asset,price,volume_24h,open_interest,trades_24h,next_funding_rate,status
0,BTC-USD,,,66284.696620,7.718559e+07,0.0,35438,-2.081818e-05,ACTIVE
1,ETH-USD,,,1975.747352,2.439335e+07,0.0,15286,-2.568864e-05,ACTIVE
5,SOL-USD,,,83.922154,1.097313e+07,0.0,6030,-2.782045e-05,ACTIVE
6,ADA-USD,,,0.276106,4.530192e+05,0.0,291,0.000000e+00,ACTIVE
16,NEAR-USD,,,1.145115,4.038262e+05,0.0,464,0.000000e+00,ACTIVE
134,PAXG-USD,,,5403.237325,3.363282e+05,0.0,414,1.093091e-04,ACTIVE
32,XRP-USD,,,1.367971,3.321802e+05,0.0,317,-5.590909e-07,ACTIVE
222,HYPE-USD,,,31.747512,3.058707e+05,0.0,679,0.000000e+00,ACTIVE
7,AVAX-USD,,,9.040070,2.647294e+05,0.0,529,0.000000e+00,ACTIVE
12,DOT-USD,,,1.531077,2.512482e+05,0.0,217,0.000000e+00,ACTIVE
